In [14]:
import pandas as pd
import os
from unidecode import unidecode

In [2]:
def combinar_h_v(df):
    """
    Combina pares de columnas que empiezan por 'h' y 'v' en un DataFrame,
    creando nuevas columnas con los valores combinados. Además, devuelve
    una lista de las columnas creadas.

    Args:
        df (pd.DataFrame): DataFrame original.

    Returns:
        pd.DataFrame: DataFrame con columnas combinadas añadidas.
        list: Lista de nombres de las columnas creadas.
    """
    columnas_h = [col for col in df.columns if col.startswith('h')]
    columnas_v = [col for col in df.columns if col.startswith('v')]
    
    columnas_creadas = []  # Lista para almacenar nombres de las columnas creadas

    for col_h, col_v in zip(columnas_h, columnas_v):
        nueva_columna = f"{col_h}_{col_v}"  # Nombre de la nueva columna
        df[nueva_columna] = df[col_h].astype(str) + " " + df[col_v].astype(str)
        columnas_creadas.append(nueva_columna)  # Agregar el nombre de la columna a la lista
    
    return df, columnas_creadas

In [3]:
def formato_df(df):
    # nos aseguramos de que las columnas de fecha son str
    df[["ano", "mes", "dia"]] = (df[["ano", "mes", "dia"]]).astype(str)
    # rellenamos mes y dia para que siempre tengan 2 dígitos
    df["mes"] = df["mes"].str.zfill(2)
    df["dia"] = df["dia"].str.zfill(2)
    # creamos la columna de fecha
    df["fecha"] = df["ano"] + "-" + df["mes"] + "-" + df["dia"]
    # combinamos las columnas de hora y validacion  
    df, columnas_creadas = combinar_h_v(df)
    # transformamos el df para que las columnas de hora queden en filas
    df = df.melt(
        id_vars = ["provincia", "municipio", "estacion", "magnitud", "punto_muestreo", "fecha"],  # Columnas que no se transforman
        value_vars = columnas_creadas,  # Columnas de horas que queremos transformar
        var_name = "hora",  # Nombre de la nueva columna que contiene las etiquetas de hora
        value_name = "valor"  # Nombre de la nueva columna que contiene los valores de las medidas
        )
    # ahora separamos la letra que acompaña al valor para crear la columna de validación y dejamos solo la hora en la columna hora
    df["hora"] = df["hora"].str.split("_", expand = True)[0].str.extract("(\\d+)")[0]
    # como tenemos un valor 24 para la hora y no lo admite datetime, lo sustituimos por 23:59 y añadimos un :00 en las demás horas
    df["hora"] = df["hora"].apply(lambda x: '23:59' if x == '24' else f"{str(x)}:00")
    # creamos una columna de fecha y hora con formato datetime para poder hacer operaciones con ella
    df["fecha_hora_f"] = pd.to_datetime(df["fecha"].astype(str) + " " + df["hora"].astype(str))
    # obtenemos el estado de validación de la medida a partir de la columna de valor
    df["validacion"] = df["valor"].str.split(" ", expand = True)[1]
    df["valor"] = df["valor"].str.split(" ", expand = True)[0]
    #creamos un id de la medida único para cada fila (estacion, contaminante, fecha y hora)
    df["id_medida"] = df["punto_muestreo"] + "_" + df["fecha"] + "_" + df["hora"] + "_" + df["validacion"]
    
    return df

In [4]:
def unir_archivos(carpeta_entrada, patron_nombre, carpeta_salida, nombre_salida):
    """
    Une los archivos de una carpera según el nombre dado y los guarda como un único csv en la carpeta de salida

    Parámetros:
        carpeta_entrada (str): ruta de la carpeta donde se encuentran los archivos que queremos unir.
        patron_nombre (str): patrón de nombre de los archivos a unir
        carpeta_salida (str): ruta de la carpeta donde se guardan los archivos unidos.
        nombre_salida (str): nombre del archivo de salida (sin extensión).

    Retorna:
        archivos: Lista con los nombres de los archivos unidos.
    """
    carpeta = carpeta_entrada
    archivos = [archivo for archivo in os.listdir(carpeta) if archivo.lower().startswith(patron_nombre.lower())]
    df_lista = [pd.read_csv(os.path.join(carpeta, archivo), sep=";", parse_dates = True, encoding="latin1") for archivo in archivos]
    print (archivos)
    df_unido = pd.concat(df_lista, ignore_index=True)
    df_unido.to_csv(f"{carpeta_salida}/{nombre_salida}.csv", index=False)
    print(f"archivo {nombre_salida}.csv creado en {carpeta_salida}")
    return df_unido

In [5]:
# unimos csv de la Comunidad de Madrid en uno solo
df_cmadrid = unir_archivos("../data/raw", "cmadrid_20", "../data/transformed", "cmadrid")

C:\Users\marta\AppData\Local\Temp\ipykernel_18500\3589473263.py:16: DtypeWarning: Columns (8,54) have mixed types. Specify dtype option on import or set low_memory=False.
  df_lista = [pd.read_csv(os.path.join(carpeta, archivo), sep=";", parse_dates = True, encoding="latin1") for archivo in archivos]


['cmadrid_2005.CSV', 'cmadrid_2006.CSV', 'cmadrid_2007.CSV', 'cmadrid_2008.CSV', 'cmadrid_2009.CSV', 'cmadrid_2010.CSV', 'cmadrid_2011.CSV', 'cmadrid_2012.CSV', 'cmadrid_2013.CSV', 'cmadrid_2014.CSV', 'cmadrid_2015.CSV', 'cmadrid_2016.CSV', 'cmadrid_2017.CSV', 'cmadrid_2018.CSV', 'cmadrid_2019.CSV', 'cmadrid_2020.CSV', 'cmadrid_2021.CSV', 'cmadrid_2022.CSV', 'cmadrid_2023.CSV', 'cmadrid_2024.CSV', 'cmadrid_2025-05-01.csv', 'cmadrid_2025-05-05.csv', 'cmadrid_2025.CSV']
archivo cmadrid.csv creado en ../data/transformed


In [6]:
df_cmadrid.head()

,provincia,municipio,estacion,magnitud,punto_muestreo,ano,mes,dia,h01,v01,...,h20,v20,h21,v21,h22,v22,h23,v23,h24,v24
0,28,102,1,1,28102001_1_38,2005,1,1,NaN,N,...,NaN,N,NaN,N,NaN,N,NaN,N,NaN,N
1,28,102,1,6,28102001_6_48,2005,1,1,NaN,N,...,NaN,N,NaN,N,NaN,N,NaN,N,NaN,N
2,28,102,1,7,28102001_7_8,2005,1,1,NaN,N,...,NaN,N,NaN,N,NaN,N,NaN,N,NaN,N
3,28,102,1,8,28102001_8_8,2005,1,1,NaN,N,...,NaN,N,NaN,N,NaN,N,NaN,N,NaN,N
4,28,102,1,10,28102001_10_49,2005,1,1,NaN,N,...,NaN,N,NaN,N,NaN,N,NaN,N,NaN,N


In [7]:
# unimos csv de  Madrid en uno solo
df_madrid = unir_archivos("../data/raw", "madrid_20", "../data/transformed", "madrid")
df_madrid.head()

['madrid_2001.csv', 'madrid_2002.csv', 'madrid_2003.csv', 'madrid_2004.csv', 'madrid_2005.csv', 'madrid_2006.csv', 'madrid_2007.csv', 'madrid_2008.csv', 'madrid_2009.csv', 'madrid_2010.csv', 'madrid_2011.csv', 'madrid_2012.csv', 'madrid_2013.csv', 'madrid_2014.csv', 'madrid_2015.csv', 'madrid_2016.csv', 'madrid_2017.csv', 'madrid_2018.csv', 'madrid_2019.csv', 'madrid_2020.csv', 'madrid_2021.csv', 'madrid_2022.csv', 'madrid_2023.csv', 'madrid_2024.csv', 'madrid_2025-05-01.csv', 'madrid_2025-05-02.csv', 'madrid_2025-05-05.csv', 'madrid_2025.csv']
archivo madrid.csv creado en ../data/transformed


,MUNICIPIO,ESTACION,MAGNITUD,PUNTO_MUESTREO,ANO,MES,DIA,H01,V01,H02,...,V20,H21,V21,H22,V22,H23,V23,H24,V24,ï»¿PROVINCIA
0,79,4,1,28079004_1_38,2001,4,1,20.0,V,28.0,...,V,11.0,V,18.0,V,27.0,V,34.0,V,NaN
1,79,4,1,28079004_1_38,2001,4,2,17.0,V,22.0,...,V,14.0,V,15.0,V,13.0,V,11.0,V,NaN
2,79,4,1,28079004_1_38,2001,4,3,11.0,V,10.0,...,V,10.0,V,11.0,V,10.0,V,9.0,V,NaN
3,79,4,1,28079004_1_38,2001,4,4,8.0,V,8.0,...,V,10.0,V,10.0,V,9.0,V,8.0,V,NaN
4,79,4,1,28079004_1_38,2001,4,5,8.0,V,8.0,...,V,9.0,V,11.0,V,13.0,V,14.0,V,NaN


In [21]:
# Transformamos el df_madrid para que tenga el mismo formato que df_cmadrid
# eliminamos duplicados
df_madrid.drop_duplicates(inplace=True)
# eliminamos una columna con muchos nulos
df_madrid = df_madrid.drop('ï»¿PROVINCIA', axis=1)
# pasamos todos los encabezdos a minúsculas
df_madrid.columns = df_madrid.columns.str.lower()
# añadimos la columna de provincia al principio
df_madrid.insert(0, "provincia", 28)
df_madrid.head(15)

,provincia,municipio,estacion,magnitud,punto_muestreo,ano,mes,dia,h01,v01,...,h20,v20,h21,v21,h22,v22,h23,v23,h24,v24
0,28,79,4,1,28079004_1_38,2001,4,1,20.0,V,...,9.0,V,11.0,V,18.0,V,27.0,V,34.0,V
1,28,79,4,1,28079004_1_38,2001,4,2,17.0,V,...,12.0,V,14.0,V,15.0,V,13.0,V,11.0,V
2,28,79,4,1,28079004_1_38,2001,4,3,11.0,V,...,9.0,V,10.0,V,11.0,V,10.0,V,9.0,V
3,28,79,4,1,28079004_1_38,2001,4,4,8.0,V,...,10.0,V,10.0,V,10.0,V,9.0,V,8.0,V
4,28,79,4,1,28079004_1_38,2001,4,5,8.0,V,...,9.0,V,9.0,V,11.0,V,13.0,V,14.0,V
5,28,79,4,1,28079004_1_38,2001,4,6,15.0,V,...,14.0,V,13.0,V,13.0,V,12.0,V,12.0,V
6,28,79,4,1,28079004_1_38,2001,4,7,10.0,V,...,11.0,V,10.0,V,11.0,V,11.0,V,9.0,V
7,28,79,4,1,28079004_1_38,2001,4,8,8.0,V,...,10.0,V,12.0,V,15.0,V,14.0,V,11.0,V
8,28,79,4,1,28079004_1_38,2001,4,9,10.0,V,...,10.0,V,15.0,V,17.0,V,20.0,V,23.0,V
9,28,79,4,1,28079004_1_38,2001,4,10,25.0,V,...,10.0,V,10.0,V,11.0,V,11.0,V,11.0,V


In [8]:
def formato_df_madrid(df):
    """
    Formatea el DataFrame de Madrid para que tenga el mismo formato que el de la Comunidad de Madrid.

    Args:
        df (pd.DataFrame): DataFrame original de Madrid.

    Returns:
        pd.DataFrame: DataFrame formateado.
    """
   
    # eliminamos duplicados
    df.drop_duplicates(inplace=True)
    # eliminamos una columna con muchos nulos
    df = df.drop('ï»¿PROVINCIA', axis=1)
    # pasamos todos los encabezdos a minúsculas
    df.columns = df.columns.str.lower()
    # añadimos la columna de provincia al principio
    df.insert(0, "provincia", 28)
    return df

In [9]:
df_madrid = formato_df_madrid(df_madrid)

In [10]:
df_madrid.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1067558 entries, 0 to 1068864
Data columns (total 56 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   provincia       1067558 non-null  int64  
 1   municipio       1067558 non-null  int64  
 2   estacion        1067558 non-null  int64  
 3   magnitud        1067558 non-null  int64  
 4   punto_muestreo  1067558 non-null  object 
 5   ano             1067558 non-null  int64  
 6   mes             1067558 non-null  int64  
 7   dia             1067558 non-null  int64  
 8   h01             1067558 non-null  float64
 9   v01             1067558 non-null  object 
 10  h02             1067558 non-null  float64
 11  v02             1067558 non-null  object 
 12  h03             1067558 non-null  float64
 13  v03             1067558 non-null  object 
 14  h04             1067558 non-null  float64
 15  v04             1067558 non-null  object 
 16  h05             1067558 non-null  float64

In [12]:
if df_madrid.columns.equals(df_cmadrid.columns):
    df_medidas = pd.concat([df_madrid, df_cmadrid], ignore_index=True)
    df_medidas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2122472 entries, 0 to 2122471
Data columns (total 56 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   provincia       int64 
 1   municipio       int64 
 2   estacion        int64 
 3   magnitud        int64 
 4   punto_muestreo  object
 5   ano             int64 
 6   mes             int64 
 7   dia             int64 
 8   h01             object
 9   v01             object
 10  h02             object
 11  v02             object
 12  h03             object
 13  v03             object
 14  h04             object
 15  v04             object
 16  h05             object
 17  v05             object
 18  h06             object
 19  v06             object
 20  h07             object
 21  v07             object
 22  h08             object
 23  v08             object
 24  h09             object
 25  v09             object
 26  h10             object
 27  v10             object
 28  h11             object
 29  v11           

In [13]:
df_medidas = combinar_h_v(df_medidas)[0]
df_medidas = formato_df(df_medidas)
df_medidas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50939328 entries, 0 to 50939327
Data columns (total 11 columns):
 #   Column          Dtype         
---  ------          -----         
 0   provincia       int64         
 1   municipio       int64         
 2   estacion        int64         
 3   magnitud        int64         
 4   punto_muestreo  object        
 5   fecha           object        
 6   hora            object        
 7   valor           object        
 8   fecha_hora_f    datetime64[ns]
 9   validacion      object        
 10  id_medida       object        
dtypes: datetime64[ns](1), int64(4), object(6)
memory usage: 4.2+ GB


In [16]:
# cargamos el csv de los datos de contaminantes
df_contaminantes = pd.read_csv("../data/raw/datos_contaminantes.csv")
df_contaminantes.head()

,CÓDIGO \r\nMAGNITUD,DESCRIPCIÓN MAGNITUD,CÓDIGO \r\nTÉCNICA \r\nDE MEDIDA,DESCRIPCIÓN TÉCNICA \r\nDE MEDIDA,UNIDAD,DESCRIPCIÓN UNIDAD
0,1,Dióxido de azufre,38,Fluorescencia ultravioleta,µg/m³,microgramos por metro cúbico
1,6,Monóxido de carbono,48,Espectrometría infrarroja no \r\ndispersiva,mg/m³,miligramos por metro cúbico
2,7,Monóxido de nitrógeno,8,Quimioluminiscencia,µg/m³,microgramos por metro cúbico
3,8,Dióxido de nitrógeno,8,Quimioluminiscencia,µg/m³,microgramos por metro cúbico
4,9,"Partículas en suspensión < PM2,5",49,Absorción beta,µg/m³,microgramos por metro cubico


In [17]:
# cambiamos el nombre de las columnas a snake_case y quitamos las tildes
df_contaminantes.columns = df_contaminantes.columns.str.lower().str.replace(" ", "_")
df_contaminantes.columns = df_contaminantes.columns.map(unidecode)
df_contaminantes.head()

,codigo_\r\nmagnitud,descripcion_magnitud,codigo_\r\ntecnica_\r\nde_medida,descripcion_tecnica_\r\nde_medida,unidad,descripcion_unidad
0,1,Dióxido de azufre,38,Fluorescencia ultravioleta,µg/m³,microgramos por metro cúbico
1,6,Monóxido de carbono,48,Espectrometría infrarroja no \r\ndispersiva,mg/m³,miligramos por metro cúbico
2,7,Monóxido de nitrógeno,8,Quimioluminiscencia,µg/m³,microgramos por metro cúbico
3,8,Dióxido de nitrógeno,8,Quimioluminiscencia,µg/m³,microgramos por metro cúbico
4,9,"Partículas en suspensión < PM2,5",49,Absorción beta,µg/m³,microgramos por metro cubico


In [ ]:
def formato_contaminantes(df):
    """
    Formatea el DataFrame de contaminantes para que tenga el formato adecuado.

    Args:
        df (pd.DataFrame): DataFrame original de contaminantes.

    Returns:
        pd.DataFrame: DataFrame formateado.
    """
    # cambiamos el nombre de las columnas a snake_case y quitamos las tildes
    df.columns = df.columns.str.lower().str.replace(" ", "_")
    df.columns = df.columns.map(unidecode)
    return df

In [ ]:
# el siguente paso es construir un df que contenga la información de todas las estaciones de medición
# para eso tenemos que unir el df de Madrid y el de la Comunidad de Madrid, que tienen más o menos las mismas columnas pero no son iguales
# tendremos que hacer algunos cambios para igualar las estructuras de ambos y poder unirlos
# primero cargamos los dos csv
estaciones_cmadrid = pd.read_csv("../data/raw/cmadrid_Red de Calidad del Aire. Estaciones.csv", sep=";", encoding="latin1")
estaciones_madrid = pd.read_csv("../data/raw/madrid_Calidad del aire. Estaciones de control.csv", sep=";")

In [ ]:
# a partir de ellos creamos un df con las zonas de medición, para eso cogemos los valores únicos de la columna "zona_calidad_aire_descripcion" de estaciones_cmadrid
df_zonas_calidad_aire = pd.DataFrame(estaciones_cmadrid["zona_calidad_aire_descripcion"].unique())
# eliminamos la palabra "Zona" de la columna
df_zonas_calidad_aire[0] = df_zonas_calidad_aire[0].str.replace("Zona ", "")
# separamos en dos columnas: zona con el número de la zona y descripcion con el nombre de la zona y eliminamos la columnna original
df_zonas_calidad_aire[["zona", "descripcion"]]= df_zonas_calidad_aire[0].str.split(" ", n=1,  expand = True)
df_zonas_calidad_aire.drop(columns=[0], inplace=True)
# añadimos la fila con la informacion de la zona de Madrid
df_zonas_calidad_aire.loc[6] = ["1", "Madrid"]